# 01 — Data Ingestion & Quality Control

**Pipeline Step 1 of 5**

Download the 10x Visium adult mouse brain dataset, apply QC filters,
normalize, and save the preprocessed AnnData.

### Outputs
- `data/processed/mouse_brain_preprocessed.h5ad`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_ingestion import get_dataset
from src.spatial_pipeline import load_adata, run_qc, normalize

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Imports ready.")

Imports ready.


## 1.1 Download Dataset

In [2]:
h5ad_path = get_dataset(DATA_RAW, source="squidpy")
adata = load_adata(h5ad_path)
print(f"\nRaw dataset: {adata.shape[0]} spots × {adata.shape[1]} genes")

/Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Dataset already cached: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/raw/mouse_brain_visium.h5ad
  Loading dataset: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/raw/mouse_brain_visium.h5ad
  Shape: 2702 spots × 32285 genes

Raw dataset: 2702 spots × 32285 genes


## 1.2 Quality Control

- Remove spots with < 200 detected genes
- Remove genes detected in < 3 spots
- Remove spots with > 30% mitochondrial content (brain tissue threshold)

In [3]:
adata = run_qc(adata)
print(f"\nPost-QC: {adata.shape[0]} spots × {adata.shape[1]} genes")

  QC filtering: 2702 → 2691 spots
  Genes retained: 19652

Post-QC: 2691 spots × 19652 genes


## 1.3 Normalization

Normalize to 10,000 counts/cell and log-transform.
Raw counts preserved in `adata.raw`.

In [4]:
adata = normalize(adata)
print(f"\nNormalized: {adata.shape[0]} spots × {adata.shape[1]} genes")
print(f"Raw layer preserved: {adata.raw is not None}")

  Normalized to 10000 counts/cell and log1p-transformed

Normalized: 2691 spots × 19652 genes
Raw layer preserved: True


## 1.4 Save Preprocessed Data

In [5]:
out_path = DATA_PROCESSED / "mouse_brain_preprocessed.h5ad"
adata.write_h5ad(out_path)
print(f"Saved: {out_path}")
print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")

Saved: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/processed/mouse_brain_preprocessed.h5ad
File size: 274.5 MB
